# 03. Análise Exploratória de Dados (EDA): Fatores de Demanda e Comparativo dos Regimes de Planejamento Humano
**Artefato Analítico-Preditivo para Planejamento de Demanda e Mitigação de Desperdício**  
*Mestrado Profissional em Educação Profissional e Tecnológica (ProfEPT) / Gestão Pública*  
**Pesquisador:** Michel Silva de Souza  

---

## Storytelling: Decifrando o Comportamento da Demanda Escolar
Antes de submeter os dados aos algoritmos de Machine Learning, é fundamental compreender a física social e institucional que rege o refeitório escolar:
1. **O Dilema dos Métodos Humanos:** Como o sistema de agendamento por reservas discentes (2023-2025) se compara ao quantitativo técnico planejado pela nutricionista (2026)?
2. **O Que Realmente Explica a Demanda?**
   - **Nível Basal e Memória Temporal:** A inércia recente como âncora principal.
   - **Calendário e Sazonalidade Acadêmica:** O efeito sistemático dos dias da semana e a desmobilização no encerramento do semestre.
   - **Engenharia de Cardápio:** O impacto de diferentes proteínas e métodos de preparo.
   - **O Papel Real do Clima:** Testando empiricamente a hipótese de que o tempo meteorológico afeta a frequência dos estudantes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

from src.core.config import GOLD, RESULTADOS
from src.core.eda import (
    carregar_dados_eda,
    avaliar_metodos_humanos,
    calcular_ranking_atributos,
    analisar_padrao_dia_semana,
    analisar_desmobilizacao_semestre,
    analisar_cardapio_proteinas,
    PASTA_EDA,
    PASTA_FIGURAS
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 130


## Bloco A: Comparativo Histórico dos Métodos Humanos de Planejamento
Comparamos os dois períodos institucionais da instituição:
* **2023 a 2025 (Reservas Discentes):** Os alunos autodeclaravam se iriam almoçar pelo aplicativo.
* **2026 (Planejamento da Nutricionista):** A nutricionista estima o quantitativo com base em sua experiência prática e contatos com coordenações.


In [ ]:
treino, periodo_discente, periodo_nutri = carregar_dados_eda()
df_humanos = avaliar_metodos_humanos(periodo_discente, periodo_nutri)
df_humanos.T


### Visualização Gráfica: Dispersão e Resíduos do Planejamento Humano
Observe como o regime discente sofre de **subnotificação crônica** (alunos que almoçam sem reservar, gerando risco de falta em 65.6% dos dias), enquanto a nutricionista opera com **superestimativa protetiva intencional** (+10.4 refeições de folga para garantir que nenhum aluno fique sem almoço).

In [ ]:
display(Image(filename=str(PASTA_FIGURAS / "figura_01_comparativo_metodos_humanos.png")))


## Bloco B: Ranking de Impacto e Correlação dos Atributos
Avaliamos a correlação de Pearson ($r$) de cada variável com o consumo diário real, desdobrada nos dois períodos e no consolidado geral.

In [ ]:
df_ranking = calcular_ranking_atributos(treino, periodo_discente, periodo_nutri)
df_ranking[["Grupo", "Atributo", "Corr_2023_2025", "Corr_2026", "Corr_Geral", "Corr_Relativa", "Descrição"]]


### Visualização: Tornado Chart de Poder Preditivo
Evidência empírica incontestável: **Nível Basal**, **Mesmo Dia da Semana** e **Dias até o Fim do Semestre** são os preditores mais fortes, enquanto as variáveis climáticas apresentam correlação linear próxima de zero.

In [ ]:
display(Image(filename=str(PASTA_FIGURAS / "figura_04_ranking_correlacoes.png")))


## Bloco C: Sazonalidade por Dia da Semana
Análise da inércia semanal: segundas e quartas concentram a maior demanda, enquanto quintas e sextas registram esvaziamento sistemático em ambos os regimes.

In [ ]:
df_semana = analisar_padrao_dia_semana(treino, periodo_discente, periodo_nutri)
df_semana


### Gráfico da Sazonalidade Semanal Comparada

In [ ]:
display(Image(filename=str(PASTA_FIGURAS / "figura_02_sazonalidade_semanal.png")))


## Bloco D: Desmobilização Acadêmica no Encerramento do Semestre
Investigamos a curva de evasão nos últimos dias letivos antes das férias escolares e recessos.

In [ ]:
df_fim = analisar_desmobilizacao_semestre(treino)
df_fim


### Gráfico da Queda de Fim de Semestre Letivo

In [ ]:
display(Image(filename=str(PASTA_FIGURAS / "figura_03_desmobilizacao_fim_semestre.png")))


## Bloco E: Atratividade do Cardápio por Tipo de Proteína
Avaliação do impacto de diferentes tipos de carnes e preparações em relação ao nível basal esperado.

In [ ]:
df_prot = analisar_cardapio_proteinas(treino)
df_prot


## Bloco F: O Mito do Clima - Sensibilidade à Chuva e Temperatura
Gráficos de dispersão com linha de tendência linear comparando previsões arquivadas de chuva e temperatura com o consumo real.

In [ ]:
display(Image(filename=str(PASTA_FIGURAS / "figura_05_sensibilidade_climatica.png")))


## Síntese para a Dissertação: Justificativa dos 4 Cenários de Modelagem

A análise exploratória estabelece os fundamentos teóricos para a definição dos cenários no torneio de modelos:

1. **Cenário 1 (Calendário + Nível Basal):** Justificado pelas correlações elevadas de `nivel_base` ($r=0.658$), dia da semana e dias até o encerramento do bloco letivo ($r=0.334$). Garante autonomia plena para o horizonte de compras ($T+2$).
2. **Cenário 2 (+ Cardápio):** Justificado pela variação significativa observada entre proteínas e preparações na adesão dos estudantes.
3. **Cenário 3 (+ Clima):** Incluído para testar se modelos não-lineares conseguem capturar interações que a correlação linear simples não detectou (ex: chuva forte combinada com sexta-feira).
4. **Cenário 4 (+ Planejamento Humano):** Justificado pelo alto poder explicativo da nutricionista em 2026 ($r=0.772$ e $WAPE=13.6\%$), demonstrando que o algoritmo deve atuar de forma híbrida e colaborativa.
